# TEP Anomaly Classification Benchmark — Fixed v3 (Paper-Replikation)
### Tennessee Eastman Process — Deep & Shallow Learning mit DR-Vergleich

**Dimensionsreduktion:** `NONE` | `PCA` | `LDA` \n
**Modelle:** LSTM-FCN · Deep CNN · TCN · RNN · LSTM · WaveNet · XGBoost · Random Forest · SVM \n
**Hyperparameter-Tuning:** GridSearchCV für XGBoost, Random Forest und SVM

> **Replikationsmodus** — Sequenzlängen (500/960), Architekturen (plain RNN/LSTM) und
> LDA-Vorverarbeitung entsprechen exakt Peter et al. 2024 (Table 1).
> Fix 7 (Pipeline) korrigiert den Implementierungsfehler im CV, ohne die Methodik zu ändern.

---
### Changelog
| # | Version | Problem | Fix |
|---|---------|---------|-----|
| 1 | v1 | **TCN RF=90 << 500** — Modell lernte nichts | `dilations=[1,2,4,8,16,32,64,128]` → RF=1530 |
| 2 | v1 | **EarlyStopping zu aggressiv** für RNN/LSTM | `patience=12` + `EPOCHS=100` für langsame Modelle |
| 3 | v1 | **SVM/XGBoost\|LDA Label-Bug** | Globaler `LabelEncoder`, einmalig auf [1..20] gefittet |
| 4 | v1 | **val_loss statt val_accuracy** | Glatterer Monitor, weniger Fehlalarme |
| 5 | v2 | **`validation_split=0.2` nicht stratifiziert** | Manueller stratifizierter Split via `train_test_split(..., stratify=y_tr)` |
| 6 | v2 | **WaveNet ohne BatchNormalization** | `BatchNormalization()` nach jedem Residual-Block eingefügt |
| 7 | **v3** | **Data Leakage in GridSearchCV** — Scaler und DR vor CV auf Gesamtdaten gefittet | `sklearn.pipeline.Pipeline(Scaler → DR → Clf)` direkt an `GridSearchCV`; Transformationen werden erst per Fold gefittet |

---
**Bewusst NICHT geändert (Paper-Replikation):**
- SEQ\_LEN\_TRAIN=500 / SEQ\_LEN\_TEST=960 (wie im Paper)
- Kein Pre-Fault-Trimming (Paper beschreibt keines)
- Plain RNN / LSTM ohne Bidirectional (Paper-Architektur)

---
**Benötigte CSV-Dateien (gleicher Ordner):**
- `TEP_Faulty_Training.csv`
- `TEP_Faulty_Testing.csv`

## 1 · Imports & Konfiguration

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline                          # FIX v3: leakagefreie CV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    LSTM, Activation, Add, BatchNormalization, Concatenate, Conv1D,
    Dense, Dropout, GlobalAveragePooling1D, Input, Multiply, SimpleRNN,
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam

try:
    from tcn import TCN
    TCN_AVAILABLE = True
    print("✓ keras-tcn verfügbar")
except ImportError:
    TCN_AVAILABLE = False
    print("⚠ keras-tcn nicht installiert → pip install keras-tcn")

print(f"TensorFlow {tf.__version__} | Pandas {pd.__version__} | NumPy {np.__version__}")


✓ keras-tcn verfügbar
TensorFlow 2.21.0 | Pandas 3.0.2 | NumPy 2.4.4


## 2 · Konstanten

In [2]:
META_COLS = ['faultNumber', 'simulationRun', 'sample']

# Sequenzlängen exakt wie im Paper (Peter et al. 2024, Section 3.3)
SEQ_LEN_TRAIN = 500
SEQ_LEN_TEST  = 960
BATCH_SIZE    = 32

# ── FIX v1-3: Globaler LabelEncoder — einmalig auf faultNumber 1..20 gefittet
GLOBAL_LE = LabelEncoder()
GLOBAL_LE.fit(np.arange(1, 21))
N_CLASSES = len(GLOBAL_LE.classes_)
print(f"Label-Klassen: {GLOBAL_LE.classes_} → {N_CLASSES} Klassen")

# ── FIX v1-2: Modell-spezifische Callbacks & Epochen ────────────────────────
def make_callbacks_fast():
    return [
        EarlyStopping(monitor='val_loss', patience=5,
                      restore_best_weights=True, mode='min'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.333,
                          patience=3, min_lr=1e-6, mode='min'),
    ]

def make_callbacks_slow():
    return [
        EarlyStopping(monitor='val_loss', patience=12,
                      restore_best_weights=True, mode='min'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.333,
                          patience=4, min_lr=1e-6, mode='min'),
    ]

EPOCHS_FAST = 50
EPOCHS_SLOW = 100

CALLBACK_MAP = {
    'LSTM-FCN': (make_callbacks_fast, EPOCHS_FAST),
    'Deep CNN': (make_callbacks_fast, EPOCHS_FAST),
    'WaveNet' : (make_callbacks_fast, EPOCHS_FAST),
    'RNN'     : (make_callbacks_slow, EPOCHS_SLOW),
    'LSTM'    : (make_callbacks_slow, EPOCHS_SLOW),
    'TCN'     : (make_callbacks_slow, EPOCHS_SLOW),
}

# ── GridSearch-Grids ──────────────────────────────────────────────────────
GRID_XGB = {
    'n_estimators' : [200, 300, 500],
    'max_depth'    : [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
}
GRID_RF = {
    'n_estimators'     : [200, 300, 500],
    'max_depth'        : [None, 10, 20],
    'min_samples_split': [2, 5],
}
GRID_SVM = {
    'C'     : [1, 10, 100],
    'gamma' : ['scale', 'auto'],
    'kernel': ['rbf'],
}


Label-Klassen: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20] → 20 Klassen


## 3 · Daten laden

Nur die **Faulty-Dateien** werden geladen (faultNumber 1–20).  
Der Benchmark fokussiert auf Anomaly Classification — FaultFree-Daten (faultNumber 0) werden nicht benötigt.

In [ ]:
def load_tep_data():
    """Lädt ausschließlich die Faulty-Dateien (faultNumber 1–20)."""
    def _load(path):
        header = pd.read_csv(path, nrows=0).columns.tolist()
        dtypes = {c: np.float32 for c in header if c not in META_COLS}
        return pd.read_csv(path, dtype=dtypes)

    df_train = _load("TEP_Faulty_Training.csv")
    df_test  = _load("TEP_Faulty_Testing.csv")

    print(f"Train : {df_train.shape} | Klassen: {sorted(df_train['faultNumber'].unique())}")
    print(f"Test  : {df_test.shape}  | Klassen: {sorted(df_test['faultNumber'].unique())}")
    return df_train, df_test

df_train, df_test = load_tep_data()

## 4 · Preprocessing-Hilfsfunktionen

In [ ]:
def _split_xy(df):
    feat = [c for c in df.columns if c not in META_COLS]
    return df[feat], df['faultNumber'], df['simulationRun']


def create_sequences(df_2d, labels_1d, runs_1d, seq_len):
    """Nicht-überlappende Sequenzen, Gruppierung nach simulationRun.

    Die Gruppierung nach simulationRun stellt sicher, dass keine
    Run-Grenzen innerhalb einer Sequenz auftreten und kein Leakage
    zwischen unterschiedlichen Simulation-Runs entsteht.
    """
    df = pd.DataFrame(df_2d)
    df['_run']   = runs_1d.values
    df['_label'] = labels_1d.values
    feat_cols = [c for c in df.columns if c not in ('_run', '_label')]
    seqs, labels = [], []
    for _, g in df.groupby('_run'):
        d, l = g[feat_cols].values, g['_label'].values
        for i in range(0, len(d) - seq_len + 1, seq_len):
            seqs.append(d[i : i + seq_len])
            labels.append(l[i + seq_len - 1])
    return np.array(seqs, dtype=np.float32), np.array(labels)


def create_aggregated(df_2d, labels_1d, runs_1d, window):
    """Mittelt Zeitfenster der Länge window → flache Feature-Vektoren."""
    df = pd.DataFrame(df_2d)
    df['_run']   = runs_1d.values
    df['_label'] = labels_1d.values
    feat_cols = [c for c in df.columns if c not in ('_run', '_label')]
    agg_X, agg_y = [], []
    for _, g in df.groupby('_run'):
        d, l = g[feat_cols].values, g['_label'].values
        for i in range(0, len(d) - window + 1, window):
            agg_X.append(d[i : i + window].mean(axis=0))
            agg_y.append(l[i + window - 1])
    return np.array(agg_X, dtype=np.float32), np.array(agg_y)


## 5 · Preprocessing-Pipelines

In [ ]:
def preprocess_deep(df_train, df_test, dr='lda'):
    """
    Skalierung → Sequenzbildung → optionale DR (pro Zeitschritt).

    Reihenfolge: Scale → Sequenz → DR
    ─────────────────────────────────
    Entspricht Peter et al. 2024, Section 3.3:
    'all data was reshaped into sequences of 500 timestamps for training
    and 960 timestamps for testing. [...] PCA and LDA were applied.'

    Sequenzlängen: SEQ_LEN_TRAIN=500, SEQ_LEN_TEST=960 (Paper-Standard).
    Kein Pre-Fault-Trimming — wie im Paper.
    """
    X_tr, y_tr, runs_tr = _split_xy(df_train)
    X_te, y_te, runs_te = _split_xy(df_test)

    # 1) Skalierung auf rohen Zeitschritten
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr).astype(np.float32)
    X_te_sc = scaler.transform(X_te).astype(np.float32)

    # 2) Sequenzbildung (kein Leakage über Run-Grenzen)
    X_tr_seq, y_tr_raw = create_sequences(X_tr_sc, y_tr, runs_tr, SEQ_LEN_TRAIN)
    X_te_seq, y_te_raw = create_sequences(X_te_sc, y_te, runs_te, SEQ_LEN_TEST)

    # 3) Optionale DR — angewendet pro Zeitschritt
    if dr in ('lda', 'pca'):
        N_tr, T_tr, F = X_tr_seq.shape
        N_te, T_te, _ = X_te_seq.shape

        X_tr_2d = X_tr_seq.reshape(-1, F)
        X_te_2d = X_te_seq.reshape(-1, F)

        if dr == 'lda':
            y_tr_rep = np.repeat(y_tr_raw, T_tr)
            red = LDA()
            X_tr_2d = red.fit_transform(X_tr_2d, y_tr_rep).astype(np.float32)
            X_te_2d = red.transform(X_te_2d).astype(np.float32)
        else:  # pca
            red = PCA(n_components=N_CLASSES - 1)
            X_tr_2d = red.fit_transform(X_tr_2d).astype(np.float32)
            X_te_2d = red.transform(X_te_2d).astype(np.float32)

        F_new = X_tr_2d.shape[1]
        X_tr_seq = X_tr_2d.reshape(N_tr, T_tr, F_new)
        X_te_seq = X_te_2d.reshape(N_te, T_te, F_new)

    # 4) Label-Encoding mit globalem Encoder
    y_tr_seq = GLOBAL_LE.transform(y_tr_raw)
    y_te_seq = GLOBAL_LE.transform(y_te_raw)

    print(f"  Deep | DR={dr.upper():<4} | Train {X_tr_seq.shape} | Test {X_te_seq.shape}")
    return X_tr_seq, y_tr_seq, X_te_seq, y_te_seq


def preprocess_shallow(df_train, df_test):
    """
    FIX v3: Skalierung und DR entfernt — beides übernimmt die sklearn
    Pipeline in train_eval_shallow, damit GridSearchCV keine geleckten
    Validierungs-Folds erhält.

    Reihenfolge: Scale(intern) → Aggregation → rohe Feature-Vektoren
    ─────────────────────────────────────────────────────────────────
    Aggregation über SEQ_LEN_TRAIN=500 / SEQ_LEN_TEST=960 Schritte
    entspricht dem Paper. Scaler und DR werden NICHT hier gefittet,
    sondern innerhalb der Pipeline in train_eval_shallow per CV-Fold.
    """
    X_tr, y_tr, runs_tr = _split_xy(df_train)
    X_te, y_te, runs_te = _split_xy(df_test)

    # Aggregation auf rohen (unskaliert) Daten — Scaler kommt in die Pipeline
    X_tr_agg, y_tr_raw = create_aggregated(X_tr, y_tr, runs_tr, SEQ_LEN_TRAIN)
    X_te_agg, y_te_raw = create_aggregated(X_te, y_te, runs_te, SEQ_LEN_TEST)

    print(f"  Shallow | Train {X_tr_agg.shape} | Test {X_te_agg.shape}")
    # Rohes y zurückgeben — Encoding übernimmt train_eval_shallow
    return X_tr_agg, y_tr_raw, X_te_agg, y_te_raw


## 6 · Modell-Definitionen

In [ ]:
def build_lstm_fcn(n_features, n_classes):
    inputs = Input(shape=(None, n_features))
    x1 = LSTM(128)(inputs)
    x1 = Dropout(0.8)(x1)
    x2 = Conv1D(128, 8, padding='same')(inputs)
    x2 = BatchNormalization()(x2); x2 = Activation('relu')(x2)
    x2 = Conv1D(256, 5, padding='same')(x2)
    x2 = BatchNormalization()(x2); x2 = Activation('relu')(x2)
    x2 = Conv1D(128, 3, padding='same')(x2)
    x2 = BatchNormalization()(x2); x2 = Activation('relu')(x2)
    x2 = GlobalAveragePooling1D()(x2)
    out = Dense(n_classes, activation='softmax')(Concatenate()([x1, x2]))
    return Model(inputs, out, name='LSTM_FCN')


def build_deep_cnn(n_features, n_classes):
    m = Sequential(name='Deep_CNN')
    m.add(Input(shape=(None, n_features)))
    for f in [64, 128, 256, 128]:
        m.add(Conv1D(f, 3, padding='same'))
        m.add(BatchNormalization())
        m.add(Activation('relu'))
    m.add(GlobalAveragePooling1D())
    m.add(Dropout(0.5))
    m.add(Dense(128, activation='relu'))
    m.add(Dense(n_classes, activation='softmax'))
    return m


def build_tcn(n_features, n_classes):
    """
    FIX v1-1: Dilationen auf [1,2,4,8,16,32,64,128] erweitert.
    Receptive Field = 2 × kernel_size × sum(dilations)
                    = 2 × 3 × 255 = 1530 >> SEQ_LEN_TRAIN=500 ✓
    """
    if not TCN_AVAILABLE:
        raise ImportError("pip install keras-tcn")
    inputs = Input(shape=(None, n_features))
    x = TCN(
        nb_filters=64,
        kernel_size=3,
        nb_stacks=2,
        dilations=[1, 2, 4, 8, 16, 32, 64, 128],
        padding='causal',
        use_skip_connections=True,
        dropout_rate=0.2,
        return_sequences=False,
    )(inputs)
    x   = Dense(64, activation='relu')(x)
    out = Dense(n_classes, activation='softmax')(x)
    return Model(inputs, out, name='TCN')


def build_rnn(n_features, n_classes):
    """Plain RNN — Paper-Architektur (Peter et al. 2024)."""
    m = Sequential(name='RNN')
    m.add(Input(shape=(None, n_features)))
    m.add(SimpleRNN(128, return_sequences=True))
    m.add(SimpleRNN(64))
    m.add(Dropout(0.3))
    m.add(Dense(n_classes, activation='softmax'))
    return m


def build_lstm(n_features, n_classes):
    """Plain LSTM — Paper-Architektur (Peter et al. 2024)."""
    m = Sequential(name='LSTM')
    m.add(Input(shape=(None, n_features)))
    m.add(LSTM(128, return_sequences=True))
    m.add(LSTM(64))
    m.add(Dropout(0.3))
    m.add(Dense(n_classes, activation='softmax'))
    return m


def build_wavenet(n_features, n_classes):
    """
    FIX v2-6: BatchNormalization nach jedem Residual-Block eingefügt.

    Architektur:
    - Kausale 1×1-Projektion + BN (Eingangsprojektion)
    - 6 Dilated-Residual-Blöcke (d=1,2,4,8,16,32) mit je:
      tanh-Gate × sigmoid-Gate → 1×1-Projektion → BN → Skip + Residual
    - Skip-Connections summieren alle Blöcke
    - 1×1-Conv + BN + GlobalAveragePooling + Dropout(0.3) + Dense
    """
    FILTERS   = 32
    DILATIONS = [1, 2, 4, 8, 16, 32]

    inputs = Input(shape=(None, n_features))

    x = Conv1D(FILTERS, 1, padding='causal')(inputs)
    x = BatchNormalization()(x)

    skips = []
    for d in DILATIONS:
        x_tanh = Conv1D(FILTERS, 2, dilation_rate=d,
                        padding='causal', activation='tanh')(x)
        x_sig  = Conv1D(FILTERS, 2, dilation_rate=d,
                        padding='causal', activation='sigmoid')(x)
        x_res  = Conv1D(FILTERS, 1)(Multiply()([x_tanh, x_sig]))
        x_res  = BatchNormalization()(x_res)
        skips.append(x_res)
        x = Add()([x, x_res])

    x   = Activation('relu')(Add()(skips))
    x   = Conv1D(FILTERS, 1, activation='relu')(x)
    x   = BatchNormalization()(x)
    x   = GlobalAveragePooling1D()(x)
    out = Dense(n_classes, activation='softmax')(Dropout(0.3)(x))
    return Model(inputs, out, name='WaveNet')


## 7 · Training & Evaluierung

In [ ]:
def train_eval_deep(model, X_tr, y_tr, X_te, y_te, name):
    """
    FIX v1-2: Modellname → CALLBACK_MAP → passendes Callback-Profil & Epochen-Limit.
    FIX v1-4: monitor='val_loss' (glatter, weniger Fehlalarme als val_accuracy).
    FIX v2-5: Stratifizierter Validation-Split statt Keras validation_split=0.2.
    """
    model_key = name.split(' | ')[0]
    cb_factory, max_epochs = CALLBACK_MAP.get(
        model_key, (make_callbacks_fast, EPOCHS_FAST)
    )
    callbacks = cb_factory()

    model.compile(
        optimizer=Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    # FIX v2-5: Stratifizierter Split — 80 % Train, 20 % Val
    X_t, X_v, y_t, y_v = train_test_split(
        X_tr, y_tr,
        test_size=0.2,
        random_state=42,
        stratify=y_tr,
    )

    is_slow     = (max_epochs == EPOCHS_SLOW)
    patience_str = '12' if is_slow else '5'

    print(f"\n{'='*62}\n {name}")
    print(f"  Fit {X_t.shape} | Val {X_v.shape} (stratifiziert) | Test {X_te.shape}")
    print(f"  Callbacks: {'SLOW' if is_slow else 'FAST'} "
          f"(patience={patience_str}, max_epochs={max_epochs})")
    print(f"{'='*62}")

    model.fit(
        X_t, y_t,
        epochs=max_epochs,
        batch_size=BATCH_SIZE,
        validation_data=(X_v, y_v),
        callbacks=callbacks,
        verbose=1,
    )

    y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
    f1 = f1_score(y_te, y_pred, average='macro')
    print(f"\n[{name}] Makro F1 = {f1:.4f}")
    print(classification_report(
        y_te, y_pred,
        target_names=[f"Fault {c}" for c in GLOBAL_LE.classes_],
        zero_division=0,
    ))
    return f1


def train_eval_shallow(model, X_tr, y_tr_raw, X_te, y_te_raw, name,
                       dr='none', param_grid=None):
    """
    FIX v3: sklearn Pipeline verhindert Data Leakage in GridSearchCV.

    Problem vorher:
        Scaler und DR wurden in preprocess_shallow auf dem gesamten
        Trainingsset gefittet.  GridSearchCV teilte danach bereits
        transformierte Daten auf — Validierungs-Folds hatten die
        Statistiken (Mittelwert, Varianz, LDA-Projektion) der gesamten
        Trainingsmenge bereits gesehen → CV-Scores waren ungültig.

    Lösung:
        Pipeline(StandardScaler → [optionale DR] → Klassifikator) wird
        direkt an GridSearchCV übergeben.  Jede Transformation wird erst
        innerhalb des jeweiligen Folds auf dessen Trainingsanteil gefittet.

    Pipeline-Aufbau je nach dr:
        'none' : StandardScaler → Klassifikator
        'pca'  : StandardScaler → PCA(n_components=19) → Klassifikator
        'lda'  : StandardScaler → LDA() → Klassifikator

    Param-Grid-Präfixierung:
        GridSearchCV erwartet '<step_name>__<param>' für Pipeline-Schritte.
        GRID_XGB['n_estimators'] wird zu 'clf__n_estimators' etc.
    """
    import itertools

    # Label-Encoding außerhalb der Pipeline (Labels sind kein Feature)
    y_tr = GLOBAL_LE.transform(y_tr_raw)
    y_te = GLOBAL_LE.transform(y_te_raw)

    # Pipeline aufbauen
    steps = [('scaler', StandardScaler())]
    if dr == 'lda':
        steps.append(('dr', LDA()))
    elif dr == 'pca':
        steps.append(('dr', PCA(n_components=N_CLASSES - 1)))
    steps.append(('clf', model))
    pipe = Pipeline(steps)

    print(f"\n{'='*62}\n {name}\n Train {X_tr.shape} | Test {X_te.shape}")

    if param_grid:
        # FIX v3: Parameternamen für die Pipeline präfixieren
        param_grid_pipe = {f'clf__{k}': v for k, v in param_grid.items()}
        n_combos = len(list(itertools.product(*param_grid_pipe.values())))
        print(f"  GridSearch läuft... ({n_combos} Kombinationen × 3 Folds)")
        gs = GridSearchCV(pipe, param_grid_pipe, cv=3,
                          scoring='f1_macro', n_jobs=-1, verbose=1)
        gs.fit(X_tr, y_tr)
        best = gs.best_estimator_
        best_params_clean = {k.replace('clf__', ''): v
                             for k, v in gs.best_params_.items()}
        print(f"  Beste Parameter : {best_params_clean}")
        print(f"  Bester CV F1    : {gs.best_score_:.4f}")
    else:
        pipe.fit(X_tr, y_tr)
        best = pipe

    print('='*62)
    y_pred = best.predict(X_te)
    f1 = f1_score(y_te, y_pred, average='macro')
    print(f"[{name}] Makro F1 = {f1:.4f}")
    print(classification_report(
        y_te, y_pred,
        target_names=[f"Fault {c}" for c in GLOBAL_LE.classes_],
        zero_division=0,
    ))
    return f1


## 8 · Haupt-Benchmark-Loop

In [ ]:
results = {}

# ── FIX v3: Shallow-Preprocessing ist DR-unabhängig → einmalig berechnen ─────
# Scaler und DR werden erst innerhalb der sklearn Pipeline in
# train_eval_shallow per Fold gefittet.  Daher muss preprocess_shallow
# nur einmal aufgerufen werden — nicht für jedes dr separat.
print("Preprocessing Shallow (einmalig, DR-unabhängig) ...")
X_tr_s_raw, y_tr_s_raw, X_te_s_raw, y_te_s_raw = preprocess_shallow(
    df_train, df_test
)

for dr in ['none', 'pca', 'lda']:
    dr_label = dr.upper()
    print(f"\n{'#'*62}\n  Dimensionsreduktion: {dr_label}\n{'#'*62}")

    # ── Deep-Branch: sequenzbasierte Modelle ──────────────────────────────────
    X_tr_d, y_tr_d, X_te_d, y_te_d = preprocess_deep(df_train, df_test, dr=dr)
    n_feat_d = X_tr_d.shape[2]

    deep_builders = [
        ('LSTM-FCN', build_lstm_fcn),
        ('Deep CNN', build_deep_cnn),
        ('TCN',      build_tcn),
        ('RNN',      build_rnn),
        ('LSTM',     build_lstm),
        ('WaveNet',  build_wavenet),
    ]
    for model_name, builder in deep_builders:
        tf.keras.backend.clear_session()
        model = builder(n_feat_d, N_CLASSES)
        key   = f"{model_name} | {dr_label}"
        results[key] = train_eval_deep(
            model, X_tr_d, y_tr_d, X_te_d, y_te_d, key
        )

    # ── Shallow-Branch: Pipeline baut Scaler + DR intern auf ─────────────────
    # FIX v3: Rohe aggregierte Features übergeben; dr-Parameter steuert,
    #         welche DR-Stufe die Pipeline enthält.
    shallow_configs = [
        (
            'XGBoost',
            XGBClassifier(random_state=42, eval_metric='mlogloss',
                          use_label_encoder=False, verbosity=0),
            GRID_XGB,
        ),
        (
            'Random Forest',
            RandomForestClassifier(random_state=42, n_jobs=-1),
            GRID_RF,
        ),
        (
            'SVM',
            SVC(random_state=42),
            GRID_SVM,
        ),
    ]
    for model_name, model, grid in shallow_configs:
        key = f"{model_name} | {dr_label}"
        results[key] = train_eval_shallow(
            model, X_tr_s_raw, y_tr_s_raw, X_te_s_raw, y_te_s_raw,
            key, dr=dr, param_grid=grid
        )


## 9 · Ergebnisübersicht & Vergleich mit Paper

In [ ]:
model_names = [
    'LSTM-FCN', 'Deep CNN', 'TCN', 'RNN', 'LSTM',
    'WaveNet', 'XGBoost', 'Random Forest', 'SVM',
]

# Paper-Referenzwerte (Peter et al. 2024, Table 1)
paper = {
    'LSTM-FCN'    : {'NONE': 0.90, 'PCA': 0.89, 'LDA': 0.98},
    'Deep CNN'    : {'NONE': 0.96, 'PCA': 0.86, 'LDA': 0.91},
    'TCN'         : {'NONE': 0.90, 'PCA': 0.89, 'LDA': 0.93},
    'RNN'         : {'NONE': 0.90, 'PCA': 0.78, 'LDA': 0.91},
    'LSTM'        : {'NONE': 0.84, 'PCA': 0.71, 'LDA': 0.88},
    'XGBoost'     : {'NONE': 0.72, 'PCA': 0.66, 'LDA': 0.74},
    'Random Forest': {'NONE': 0.70, 'PCA': 0.62, 'LDA': 0.77},
    'SVM'         : {'NONE': 0.62, 'PCA': 0.60, 'LDA': 0.73},
    'WaveNet'     : {'NONE': 0.62, 'PCA': 0.44, 'LDA': 0.66},
}

DR_COLS = ['NONE', 'PCA', 'LDA']

print("\n" + "="*90)
print(f"  {'Modell':<16}  {'NONE':>7} {'PCA':>7} {'LDA':>7} {'AVG':>7}  "
      f"{'Δ NONE':>8} {'Δ PCA':>8} {'Δ LDA':>8}")
print("-"*90)

for m in model_names:
    vals   = [results.get(f"{m} | {dr}", float('nan')) for dr in DR_COLS]
    p_vals = [paper.get(m, {}).get(dr, float('nan'))    for dr in DR_COLS]
    deltas = [
        v - pv if not (np.isnan(v) or np.isnan(pv)) else float('nan')
        for v, pv in zip(vals, p_vals)
    ]
    avg = np.nanmean(vals)

    row = f"  {m:<16} "
    row += "".join(f" {v:>7.4f}" if not np.isnan(v) else f" {'—':>7}" for v in vals)
    row += f" {avg:>7.4f} "
    row += "".join(f" {d:>+8.4f}" if not np.isnan(d) else f" {'—':>8}" for d in deltas)
    print(row)

print("="*90)
print("\nDelta = Eigener Wert − Paper-Referenzwert  (+ besser, − schlechter als Paper)")

print("\n" + "─"*50)
print("Paper-Referenzwerte (Peter et al. 2024):")
print(f"  {'Modell':<16}  {'NONE':>7} {'PCA':>7} {'LDA':>7} {'AVG':>7}")
print("─"*50)
for m in model_names:
    p = paper.get(m, {})
    pv = [p.get(dr, float('nan')) for dr in DR_COLS]
    avg_p = np.nanmean(pv)
    row = f"  {m:<16} "
    row += "".join(f" {v:>7.2f}" if not np.isnan(v) else f" {'—':>7}" for v in pv)
    row += f" {avg_p:>7.2f}"
    print(row)
print("─"*50)